# 06 - Entrainement complet des modeles

Ce notebook regenere tout le pipeline final : preparation des donnees, analyse du desequilibre, baseline, comparaison des techniques de reequilibrage, validation croisee stratifiee, optimisation du seuil, sauvegarde des modeles et rapports.

La logique metier est prioritaire : dans un contexte de retention, rater un churner coute plus cher que contacter un client en trop. On suit donc surtout le recall, tout en surveillant la precision, le F1-score, la ROC-AUC et la PR-AUC.


## 1. Imports et chemins


In [1]:
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
import sys
sys.path.append(str(BASE_DIR))
from feature_engineering import prepare_customer_features

DATA_PATH = BASE_DIR / 'data' / 'customer_churn.csv'
MODELS_DIR = BASE_DIR / 'models'
REPORTS_DIR = BASE_DIR / 'reports'
PREPROCESSED_PATH = BASE_DIR / 'data_preprocessed.pkl'

RANDOM_STATE = 42
DEFAULT_THRESHOLD = 0.50
THRESHOLD_GRID = np.round(np.arange(0.10, 0.91, 0.05), 2)
MIN_PRECISION_FOR_RECALL = 0.20
MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)


## 2. Chargement et feature engineering

On applique les variables metier creees dans `feature_engineering.py` : satisfaction, engagement, risque paiement, pression support, etc.


In [2]:
raw_df = pd.read_csv(DATA_PATH)
df = prepare_customer_features(raw_df)

X = df.drop(columns=['churn', 'customer_id'])
y = df['churn']

print('Shape X :', X.shape)
print('Taux de churn :', round(y.mean(), 4))


Shape X : (10000, 41)
Taux de churn : 0.1021


## 3. Analyse prealable du desequilibre

La cible est fortement desequilibree. Il faut donc regarder le ratio majoritaire/minoritaire et ne pas se limiter a l'accuracy. Un modele qui predit presque toujours `pas de churn` peut avoir une accuracy elevee, mais il ne sert pas le besoin metier si les vrais churners ne sont pas detectes.


In [3]:
class_counts = y.value_counts().sort_index()
majority_class = class_counts.idxmax()
minority_class = class_counts.idxmin()
imbalance_ratio = class_counts.max() / class_counts.min()

imbalance_summary = pd.DataFrame({
    'classe': class_counts.index,
    'effectif': class_counts.values,
    'proportion': (class_counts / len(y)).values,
})

print('Classe majoritaire :', majority_class)
print('Classe minoritaire :', minority_class)
print('Ratio majoritaire/minoritaire :', round(imbalance_ratio, 2))
imbalance_summary


Classe majoritaire : 0
Classe minoritaire : 1
Ratio majoritaire/minoritaire : 8.79


,classe,effectif,proportion
0,0,8979,0.8979
1,1,1021,0.1021


## 4. Split train/test stratifie

Le split est stratifie pour conserver la meme proportion de churners dans le train et dans le test. C'est indispensable avec une classe minoritaire autour de 10 %.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
train_ratio = y_train.value_counts().max() / y_train.value_counts().min()

print('Train :', X_train.shape)
print('Test :', X_test.shape)
print('Churn train :', round(y_train.mean(), 4))
print('Churn test :', round(y_test.mean(), 4))
print('Ratio train majoritaire/minoritaire :', round(train_ratio, 2))


Train : (8000, 41)
Test : (2000, 41)
Churn train : 0.1021
Churn test : 0.102
Ratio train majoritaire/minoritaire : 8.79


## 5. Preprocessing adapte aux modeles

La regression logistique et le MLP utilisent un `StandardScaler`, car ils sont sensibles aux echelles. Les modeles d'arbres gardent les variables numeriques telles quelles.


In [5]:
def make_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

pre_scaled = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', make_encoder(), cat_cols),
])

pre_unscaled = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', make_encoder(), cat_cols),
])


## 6. Baseline simple a seuil 0.5

La baseline sert a montrer pourquoi l'accuracy ne suffit pas. On entraine une regression logistique sans reequilibrage et on l'evalue au seuil par defaut `0.5`, puis on compare aussi avec un modele naif qui predit toujours la classe majoritaire.


In [6]:
def metric_row(name, strategy, y_true, y_pred, scores=None, threshold=DEFAULT_THRESHOLD):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    row = {
        'modele': name,
        'strategie_desequilibre': strategy,
        'threshold': threshold,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
        'clients_alertes': int(y_pred.sum()),
    }
    if scores is not None:
        row['roc_auc'] = roc_auc_score(y_true, scores)
        row['pr_auc'] = average_precision_score(y_true, scores)
    else:
        row['roc_auc'] = np.nan
        row['pr_auc'] = np.nan
    return row

naive = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
naive.fit(X_train, y_train)
naive_pred = naive.predict(X_test)

baseline_lr = Pipeline([
    ('pre', pre_scaled),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
baseline_lr.fit(X_train, y_train)
baseline_scores = baseline_lr.predict_proba(X_test)[:, 1]
baseline_pred = (baseline_scores >= DEFAULT_THRESHOLD).astype(int)

baseline_rows = [
    metric_row('NaiveMajority', 'baseline_majoritaire', y_test, naive_pred, None, DEFAULT_THRESHOLD),
    metric_row('LogisticRegression_baseline', 'aucun_reequilibrage', y_test, baseline_pred, baseline_scores, DEFAULT_THRESHOLD),
]
baseline_report = pd.DataFrame(baseline_rows)
baseline_report.to_csv(REPORTS_DIR / 'baseline_analysis.csv', index=False)
baseline_report


,modele,strategie_desequilibre,threshold,accuracy,precision,recall,f1,tn,fp,fn,tp,clients_alertes,roc_auc,pr_auc
0,NaiveMajority,baseline_majoritaire,0.5,0.8980,0.000000,0.000000,0.000000,1796,0,204,0,0,NaN,NaN
1,LogisticRegression_baseline,aucun_reequilibrage,0.5,0.8955,0.413793,0.058824,0.103004,1779,17,192,12,29,0.747691,0.251169


## 7. Experiences de gestion du desequilibre

On compare explicitement plusieurs approches : aucune correction, reechantillonnage data-level (`RandomOverSampler`, `SMOTE`, `RandomUnderSampler`) et approches model-level (`class_weight`, `scale_pos_weight`).

Tous les reechantillonnages sont places dans des pipelines `imblearn`, apres le preprocessing, pour eviter toute fuite de donnees pendant la validation croisee.


In [7]:
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

experiments = {
    'LogisticRegression_baseline': {
        'strategy': 'aucun_reequilibrage',
        'pipeline': Pipeline([
            ('pre', pre_scaled),
            ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]),
    },
    'LogisticRegression_class_weight': {
        'strategy': 'class_weight_balanced',
        'pipeline': Pipeline([
            ('pre', pre_scaled),
            ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)),
        ]),
    },
    'LogisticRegression_random_over': {
        'strategy': 'random_over_sampling',
        'pipeline': Pipeline([
            ('pre', pre_scaled),
            ('sampler', RandomOverSampler(random_state=RANDOM_STATE)),
            ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]),
    },
    'LogisticRegression_smote': {
        'strategy': 'smote',
        'pipeline': Pipeline([
            ('pre', pre_scaled),
            ('sampler', SMOTE(random_state=RANDOM_STATE)),
            ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]),
    },
    'LogisticRegression_random_under': {
        'strategy': 'random_under_sampling',
        'pipeline': Pipeline([
            ('pre', pre_scaled),
            ('sampler', RandomUnderSampler(random_state=RANDOM_STATE)),
            ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]),
    },
    'RandomForest_baseline': {
        'strategy': 'aucun_reequilibrage',
        'pipeline': Pipeline([
            ('pre', pre_unscaled),
            ('model', RandomForestClassifier(
                n_estimators=250,
                max_depth=10,
                min_samples_leaf=2,
                n_jobs=1,
                random_state=RANDOM_STATE,
            )),
        ]),
    },
    'RandomForest_class_weight': {
        'strategy': 'class_weight_balanced_subsample',
        'pipeline': Pipeline([
            ('pre', pre_unscaled),
            ('model', RandomForestClassifier(
                n_estimators=250,
                max_depth=10,
                min_samples_leaf=2,
                class_weight='balanced_subsample',
                n_jobs=1,
                random_state=RANDOM_STATE,
            )),
        ]),
    },
    'XGBoost_scale_pos_weight': {
        'strategy': 'scale_pos_weight',
        'pipeline': Pipeline([
            ('pre', pre_unscaled),
            ('model', XGBClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                scale_pos_weight=scale_pos_weight,
                eval_metric='logloss',
                random_state=RANDOM_STATE,
            )),
        ]),
    },
    'DeepLearning_smote': {
        'strategy': 'smote',
        'pipeline': Pipeline([
            ('pre', pre_scaled),
            ('sampler', SMOTE(random_state=RANDOM_STATE)),
            ('model', MLPClassifier(
                hidden_layer_sizes=(64, 32),
                alpha=0.001,
                early_stopping=True,
                max_iter=300,
                random_state=RANDOM_STATE,
            )),
        ]),
    },
}

print('scale_pos_weight XGBoost :', round(scale_pos_weight, 2))
print("Nombre d'experiences :", len(experiments))


scale_pos_weight XGBoost : 8.79
Nombre d'experiences : 9


## 8. Evaluation avec validation croisee stratifiee

`StratifiedKFold` preserve la proportion des classes dans chaque fold. C'est le bon choix ici car une validation non stratifiee pourrait produire des folds avec trop peu de churners, ce qui rendrait le recall et la PR-AUC instables.


In [8]:
def evaluate_experiment(name, config):
    model = config['pipeline']
    strategy = config['strategy']
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scoring = {
        'roc_auc': 'roc_auc',
        'pr_auc': 'average_precision',
        'recall': 'recall',
        'f1': 'f1',
        'precision': 'precision',
    }

    start = perf_counter()
    cv_scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        error_score='raise',
    )
    model.fit(X_train, y_train)
    duration = perf_counter() - start

    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= DEFAULT_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    return model, {
        'modele': name,
        'strategie_desequilibre': strategy,
        'cv_roc_auc_mean': cv_scores['test_roc_auc'].mean(),
        'cv_pr_auc_mean': cv_scores['test_pr_auc'].mean(),
        'cv_recall_mean': cv_scores['test_recall'].mean(),
        'cv_f1_mean': cv_scores['test_f1'].mean(),
        'cv_precision_mean': cv_scores['test_precision'].mean(),
        'test_threshold_default': DEFAULT_THRESHOLD,
        'test_roc_auc': roc_auc_score(y_test, proba),
        'test_pr_auc': average_precision_score(y_test, proba),
        'test_accuracy': accuracy_score(y_test, pred),
        'test_precision': precision_score(y_test, pred, zero_division=0),
        'test_recall': recall_score(y_test, pred),
        'test_f1': f1_score(y_test, pred, zero_division=0),
        'test_tn': int(tn),
        'test_fp': int(fp),
        'test_fn': int(fn),
        'test_tp': int(tp),
        'test_clients_alertes': int(pred.sum()),
        'temps_entrainement_secondes': duration,
    }

rows = []
trained_models = {}

for name, config in experiments.items():
    print('Entrainement :', name, '-', config['strategy'])
    fitted_model, row = evaluate_experiment(name, config)
    trained_models[name] = fitted_model
    rows.append(row)
    joblib.dump(fitted_model, MODELS_DIR / f'model_{name}.pkl')

comparison_default = pd.DataFrame(rows).sort_values(
    ['test_recall', 'test_pr_auc', 'test_f1'], ascending=False
)
comparison_default


Entrainement : LogisticRegression_baseline - aucun_reequilibrage
Entrainement : LogisticRegression_class_weight - class_weight_balanced
Entrainement : LogisticRegression_random_over - random_over_sampling
Entrainement : LogisticRegression_smote - smote
Entrainement : LogisticRegression_random_under - random_under_sampling
Entrainement : RandomForest_baseline - aucun_reequilibrage
Entrainement : RandomForest_class_weight - class_weight_balanced_subsample
Entrainement : XGBoost_scale_pos_weight - scale_pos_weight
Entrainement : DeepLearning_smote - smote


,modele,strategie_desequilibre,cv_roc_auc_mean,cv_pr_auc_mean,cv_recall_mean,cv_f1_mean,cv_precision_mean,test_threshold_default,test_roc_auc,test_pr_auc,test_accuracy,test_precision,test_recall,test_f1,test_tn,test_fp,test_fn,test_tp,test_clients_alertes,temps_entrainement_secondes
7,XGBoost_scale_pos_weight,scale_pos_weight,0.783544,0.262367,0.533631,0.356336,0.267516,0.5,0.796615,0.281602,0.7745,0.267420,0.696078,0.386395,1407,389,62,142,531,1.590310
1,LogisticRegression_class_weight,class_weight_balanced,0.757199,0.249929,0.670765,0.325578,0.214961,0.5,0.750033,0.251288,0.6880,0.197406,0.671569,0.305122,1239,557,67,137,694,0.312310
2,LogisticRegression_random_over,random_over_sampling,0.755413,0.248444,0.662187,0.321535,0.212330,0.5,0.746015,0.246112,0.6925,0.199122,0.666667,0.306652,1249,547,68,136,683,0.376364
4,LogisticRegression_random_under,random_under_sampling,0.737246,0.235483,0.668292,0.306761,0.199080,0.5,0.739440,0.244048,0.6830,0.192857,0.661765,0.298673,1231,565,69,135,700,0.267507
3,LogisticRegression_smote,smote,0.752987,0.249485,0.651166,0.317309,0.209787,0.5,0.741231,0.246818,0.6810,0.190000,0.651961,0.294248,1229,567,71,133,700,2.796886
6,RandomForest_class_weight,class_weight_balanced_subsample,0.795146,0.267968,0.157873,0.204125,0.291087,0.5,0.798002,0.262276,0.8530,0.315574,0.377451,0.343750,1629,167,127,77,244,9.133355
8,DeepLearning_smote,smote,0.645097,0.175876,0.205626,0.207364,0.209146,0.5,0.699318,0.207502,0.8370,0.241525,0.279412,0.259091,1617,179,147,57,236,11.741627
0,LogisticRegression_baseline,aucun_reequilibrage,0.754818,0.247464,0.028146,0.051790,0.327381,0.5,0.747691,0.251169,0.8955,0.413793,0.058824,0.103004,1779,17,192,12,29,0.332540
5,RandomForest_baseline,aucun_reequilibrage,0.803621,0.298235,0.002446,0.004875,0.666667,0.5,0.805013,0.324355,0.8980,0.000000,0.000000,0.000000,1796,0,204,0,0,8.123850


## 9. Optimisation du seuil de decision

Le seuil `0.5` n'est pas optimal dans un contexte desequilibre. On teste plusieurs seuils pour chaque modele et on observe le compromis : baisser le seuil augmente souvent le recall, mais augmente aussi les faux positifs.


In [9]:
threshold_rows = []
for name, model in trained_models.items():
    proba = model.predict_proba(X_test)[:, 1]
    strategy = experiments[name]['strategy']
    for threshold in THRESHOLD_GRID:
        pred = (proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
        threshold_rows.append({
            'modele': name,
            'strategie_desequilibre': strategy,
            'threshold': threshold,
            'recall': recall_score(y_test, pred),
            'precision': precision_score(y_test, pred, zero_division=0),
            'f1': f1_score(y_test, pred, zero_division=0),
            'accuracy': accuracy_score(y_test, pred),
            'roc_auc': roc_auc_score(y_test, proba),
            'pr_auc': average_precision_score(y_test, proba),
            'tn': int(tn),
            'fp': int(fp),
            'fn': int(fn),
            'tp': int(tp),
            'clients_alertes': int(pred.sum()),
        })

thresholds = pd.DataFrame(threshold_rows)
best_f1_by_model = thresholds.loc[thresholds.groupby('modele')['f1'].idxmax()].copy()
recall_candidates = thresholds[thresholds['precision'] >= MIN_PRECISION_FOR_RECALL].copy()
if recall_candidates.empty:
    recall_candidates = thresholds.copy()
best_recall_tradeoff = recall_candidates.sort_values(
    ['recall', 'precision', 'f1', 'pr_auc'], ascending=False
).iloc[0]

print('Meilleur seuil metier :')
print(best_recall_tradeoff[['modele', 'strategie_desequilibre', 'threshold', 'recall', 'precision', 'f1', 'fp', 'fn', 'tp', 'tn']])

thresholds_sorted = thresholds.sort_values(['recall', 'precision', 'f1'], ascending=False)
thresholds_sorted.head(15)


Meilleur seuil metier :
modele                    XGBoost_scale_pos_weight
strategie_desequilibre            scale_pos_weight
threshold                                      0.2
recall                                    0.882353
precision                                 0.210773
f1                                        0.340265
fp                                             674
fn                                              24
tp                                             180
tn                                            1122
Name: 121, dtype: object


,modele,strategie_desequilibre,threshold,recall,precision,f1,accuracy,roc_auc,pr_auc,tn,fp,fn,tp,clients_alertes
34,LogisticRegression_random_over,random_over_sampling,0.10,1.000000,0.104562,0.189327,0.1265,0.746015,0.246112,49,1747,0,204,1951
17,LogisticRegression_class_weight,class_weight_balanced,0.10,1.000000,0.104508,0.189239,0.1260,0.750033,0.251288,48,1748,0,204,1952
68,LogisticRegression_random_under,random_under_sampling,0.10,1.000000,0.104401,0.189064,0.1250,0.739440,0.244048,46,1750,0,204,1954
102,RandomForest_class_weight,class_weight_balanced_subsample,0.10,1.000000,0.102256,0.185539,0.1045,0.798002,0.262276,5,1791,0,204,1995
69,LogisticRegression_random_under,random_under_sampling,0.15,0.990196,0.108544,0.195642,0.1695,0.739440,0.244048,137,1659,2,202,1861
18,LogisticRegression_class_weight,class_weight_balanced,0.15,0.985294,0.108707,0.195811,0.1745,0.750033,0.251288,148,1648,3,201,1849
51,LogisticRegression_smote,smote,0.10,0.985294,0.106181,0.191702,0.1525,0.741231,0.246818,104,1692,3,201,1893
35,LogisticRegression_random_over,random_over_sampling,0.15,0.975490,0.107976,0.194431,0.1755,0.746015,0.246112,152,1644,5,199,1843
103,RandomForest_class_weight,class_weight_balanced_subsample,0.15,0.970588,0.120511,0.214402,0.2745,0.798002,0.262276,351,1445,6,198,1643
70,LogisticRegression_random_under,random_under_sampling,0.20,0.965686,0.114535,0.204782,0.2350,0.739440,0.244048,273,1523,7,197,1720


## 10. Comparaison finale et sauvegardes

Le modele final est choisi selon le recall sous contrainte de precision minimale. Cette regle garde l'objectif industriel principal (minimiser les faux negatifs) tout en evitant une solution qui declenche trop d'alertes inutiles.


In [10]:
final_name = best_recall_tradeoff['modele']
recommended_threshold = float(best_recall_tradeoff['threshold'])

final_threshold_metrics = thresholds[thresholds['modele'].eq(final_name) & thresholds['threshold'].eq(recommended_threshold)].iloc[0]
comparison = comparison_default.merge(
    best_f1_by_model[['modele', 'threshold', 'recall', 'precision', 'f1', 'accuracy', 'fp', 'fn', 'tp', 'tn']].rename(columns={
        'threshold': 'best_f1_threshold',
        'recall': 'best_f1_recall',
        'precision': 'best_f1_precision',
        'f1': 'best_f1',
        'accuracy': 'best_f1_accuracy',
        'fp': 'best_f1_fp',
        'fn': 'best_f1_fn',
        'tp': 'best_f1_tp',
        'tn': 'best_f1_tn',
    }),
    on='modele',
    how='left',
)
comparison['modele_final'] = comparison['modele'].eq(final_name)
comparison['threshold_recommande'] = np.where(comparison['modele_final'], recommended_threshold, np.nan)
comparison = comparison.sort_values(['modele_final', 'test_recall', 'test_pr_auc', 'test_f1'], ascending=False)

comparison.to_csv(REPORTS_DIR / 'model_comparison.csv', index=False)
thresholds_sorted.to_csv(REPORTS_DIR / 'threshold_analysis.csv', index=False)

model_aliases = {
    'model_LogisticRegression.pkl': 'LogisticRegression_class_weight',
    'model_RandomForest.pkl': 'RandomForest_class_weight',
    'model_XGBoost.pkl': 'XGBoost_scale_pos_weight',
    'model_DeepLearning.pkl': 'DeepLearning_smote',
}
for filename, source_name in model_aliases.items():
    joblib.dump(trained_models[source_name], MODELS_DIR / filename)

joblib.dump(trained_models[final_name], MODELS_DIR / 'best_model.pkl')

data_info = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'num_cols': num_cols,
    'cat_cols': cat_cols,
    'all_cols': X.columns.tolist(),
    'input_cols': raw_df.drop(columns=['churn', 'customer_id']).columns.tolist(),
    'medians': X.median(numeric_only=True).to_dict(),
    'modes': X.mode(dropna=True).iloc[0].to_dict(),
    'threshold_recommande': recommended_threshold,
    'modele_retenu': final_name,
    'strategie_retenue': experiments[final_name]['strategy'],
    'ratio_desequilibre': float(imbalance_ratio),
    'min_precision_for_recall': MIN_PRECISION_FOR_RECALL,
}
joblib.dump(data_info, PREPROCESSED_PATH)

print('Modele retenu :', final_name)
print('Strategie retenue :', experiments[final_name]['strategy'])
print('Seuil recommande :', recommended_threshold)
print('Fichiers sauvegardes dans :', REPORTS_DIR, MODELS_DIR)
comparison


Modele retenu : XGBoost_scale_pos_weight
Strategie retenue : scale_pos_weight
Seuil recommande : 0.2
Fichiers sauvegardes dans : c:\Users\Héctor\OneDrive\Desktop\Projects\EFREI\M1-DE\PRO_DATA_SCIENCE\churn_predict\projet_churn_structure\reports c:\Users\Héctor\OneDrive\Desktop\Projects\EFREI\M1-DE\PRO_DATA_SCIENCE\churn_predict\projet_churn_structure\models


,modele,strategie_desequilibre,cv_roc_auc_mean,cv_pr_auc_mean,cv_recall_mean,cv_f1_mean,cv_precision_mean,test_threshold_default,test_roc_auc,test_pr_auc,...,best_f1_recall,best_f1_precision,best_f1,best_f1_accuracy,best_f1_fp,best_f1_fn,best_f1_tp,best_f1_tn,modele_final,threshold_recommande
0,XGBoost_scale_pos_weight,scale_pos_weight,0.783544,0.262367,0.533631,0.356336,0.267516,0.5,0.796615,0.281602,...,0.696078,0.267420,0.386395,0.7745,389,62,142,1407,True,0.2
1,LogisticRegression_class_weight,class_weight_balanced,0.757199,0.249929,0.670765,0.325578,0.214961,0.5,0.750033,0.251288,...,0.328431,0.335000,0.331683,0.8650,133,137,67,1663,False,NaN
2,LogisticRegression_random_over,random_over_sampling,0.755413,0.248444,0.662187,0.321535,0.212330,0.5,0.746015,0.246112,...,0.387255,0.283154,0.327122,0.8375,200,125,79,1596,False,NaN
3,LogisticRegression_random_under,random_under_sampling,0.737246,0.235483,0.668292,0.306761,0.199080,0.5,0.739440,0.244048,...,0.617647,0.216495,0.320611,0.7330,456,78,126,1340,False,NaN
4,LogisticRegression_smote,smote,0.752987,0.249485,0.651166,0.317309,0.209787,0.5,0.741231,0.246818,...,0.544118,0.231250,0.324561,0.7690,369,93,111,1427,False,NaN
5,RandomForest_class_weight,class_weight_balanced_subsample,0.795146,0.267968,0.157873,0.204125,0.291087,0.5,0.798002,0.262276,...,0.808824,0.259027,0.392390,0.7445,472,39,165,1324,False,NaN
6,DeepLearning_smote,smote,0.645097,0.175876,0.205626,0.207364,0.209146,0.5,0.699318,0.207502,...,0.406863,0.253049,0.312030,0.8170,245,121,83,1551,False,NaN
7,LogisticRegression_baseline,aucun_reequilibrage,0.754818,0.247464,0.028146,0.051790,0.327381,0.5,0.747691,0.251169,...,0.504902,0.235698,0.321373,0.7825,334,101,103,1462,False,NaN
8,RandomForest_baseline,aucun_reequilibrage,0.803621,0.298235,0.002446,0.004875,0.666667,0.5,0.805013,0.324355,...,0.553922,0.298153,0.387650,0.8215,266,91,113,1530,False,NaN


## 11. Discussion metier

Les methodes data-level permettent d'augmenter la detection des churners, mais elles ont des limites :

- `RandomOverSampler` duplique des exemples minoritaires et peut favoriser l'overfitting.
- `SMOTE` cree des exemples synthetiques utiles, mais peut ajouter du bruit si les voisins minoritaires sont mal separes.
- `RandomUnderSampler` reduit la classe majoritaire et peut perdre de l'information.
- Les ponderations (`class_weight`, `scale_pos_weight`) gardent toutes les donnees et rendent les erreurs sur churners plus couteuses.

Le seuil final traduit le compromis metier : un seuil plus bas diminue les faux negatifs, mais augmente les faux positifs. En retention client, ce compromis est acceptable si le cout d'une action commerciale reste inferieur au cout d'un client perdu.
